In [ ]:
import tiktoken
import pandas as pd

## Count the total tokens in a CSV file
def count_tokens_in_csv(file_path: str, encoding_name: str = "cl100k_base", max_length: int = 8192) -> int:
    encoding = tiktoken.get_encoding(encoding_name)

    # Load CSV
    df = pd.read_csv(file_path)

    total_tokens = 0

    for idx, code_snippet in enumerate(df['code']):
        if not isinstance(code_snippet, str):  # Handle NaN or non-string values
            continue

        num_tokens = len(encoding.encode(code_snippet))

        if num_tokens > max_length:
            print(f"Row {idx} has {num_tokens} tokens (exceeds {max_length})")

        total_tokens += num_tokens

    print(f"Total tokens in {file_path}: {total_tokens}")
    return total_tokens


train_total = count_tokens_in_csv("data/train_24892.csv")

## Request from OpenAI

In [ ]:
import os
import random
import openai
from scipy.spatial import distance
from sklearn.cluster import KMeans
import tiktoken
import pandas as pd
import time
import torch
from dotenv import load_dotenv

load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def clean_and_truncate(text):
    MAX_TOKENS = 8192
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    if len(tokens) > MAX_TOKENS:
        print(f"Truncating text from {len(tokens)} to {MAX_TOKENS} tokens.")
        return encoding.decode(tokens[:MAX_TOKENS])
    return text
    
def get_embeddings_batch(text_list, model_name):
    cleaned_inputs = [clean_and_truncate(t) for t in text_list]

    for delay_secs in (2**x for x in range(0, 6)):
        try:
            response = client.embeddings.create(
                model=model_name,
                input=cleaned_inputs
            )
            return [r.embedding for r in response.data]

        except openai.OpenAIError as e:
            # Check for unrecoverable errors
            message = str(e).lower()
            if any(term in message for term in [
                "insufficient_quota",
                "billing_hard_limit_reached",
                "account_deactivated",
                "not enough credits",
            ]):
                print(f"Fatal error detected: {e}")
                raise RuntimeError(f"Fatal error, aborting further calls: {e}") from e

            # Otherwise, retry with backoff
            sleep_time = delay_secs + random.uniform(0, 1)
            print(f"Error: {e}. Retrying in {round(sleep_time, 2)} seconds.")
            time.sleep(sleep_time)

    # After all retries fail
    print("All retries failed. Returning None embeddings for this batch.")
    return [None] * len(text_list)




def embed_dataframe_column(df, output_file, model, column="code", batch_size=100):
    """
    Embeds a dataframe column and saves embeddings to disk.
    Resumes from existing embeddings if output_file already exists.
    """
    texts = df[column].astype(str).tolist()
    all_embeddings = []

    # Check if file already exists
    if os.path.exists(output_file):
        print(f"Loading existing embeddings from {output_file}")
        embedding_tensor = torch.load(output_file)
        all_embeddings = embedding_tensor.tolist()
        start_idx = len(all_embeddings)
        print(f"Resuming from index {start_idx}")
    else:
        start_idx = 0

    # Process remaining batches
    for i in range(start_idx, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        
        embeddings = get_embeddings_batch(batch, model)
        all_embeddings.extend(embeddings)

        # Optional: Save after each batch to avoid losing progress
        embedding_tensor = torch.tensor(all_embeddings)
        torch.save(embedding_tensor, output_file)

        print(f"Processed and saved batch: {i + len(batch)} / {len(texts)}. "
              f"Size of tensor so far: {embedding_tensor.size()}")

    # Final save (in case you skipped batch saves)
    torch.save(torch.tensor(all_embeddings), output_file)
    print(f"All embeddings saved to {output_file}")

    # Add embeddings to the dataframe (optional)
    # Ensure same length
    if len(all_embeddings) == len(df):
        df["embedding"] = all_embeddings
    else:
        print("Warning: Number of embeddings does not match dataframe rows. Skipping assignment to DataFrame.")

    return df


## Rosetta Code

In [ ]:
test_df = pd.read_csv('data/test_13786.csv')
desc_df = pd.read_csv('data/test_532.csv')
train_df =pd.read_csv('data/train_24892.csv')
data = [test_df, desc_df, train_df]

In [ ]:
output_file = ['out/test/embedding_ada_13786.pt', 'out/desc/embedding_ada_532.pt', 'out/train/embedding_ada_24892.pt']
model_name = "text-embedding-ada-002"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)

In [ ]:
output_file = ['out/test/embedding_3_small_13786.pt', 'out/desc/embedding_3_small_532.pt', 'out/train/embedding_3_small_24892.pt']
model_name = "text-embedding-3-small"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)

In [ ]:
output_file = ['out/test/embedding_3_large_13786.pt', 'out/desc/embedding_3_large_532.pt', 'out/train/embedding_3_large_24892.pt']
model_name = "text-embedding-3-large"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)

## CodeNet 

In [ ]:
test_df = pd.read_csv('codenet/test_1824.csv')
desc_df = pd.read_csv('codenet/test_55.csv')
train_df =pd.read_csv('codenet/train_1725993.csv')
data = [test_df, desc_df, train_df]

In [ ]:
output_file = ['out/test/embedding_ada_1824.pt', 'out/desc/embedding_ada_55.pt', 'out/train/embedding_ada_1725993.pt']
model_name = "text-embedding-ada-002"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)

In [ ]:
output_file = ['out/test/embedding_3_small_1824.pt', 'out/desc/embedding_3_small_55.pt', 'out/train/embedding_3_small_1725993.pt']
model_name = "text-embedding-3-small"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)

In [ ]:
output_file = ['out/test/embedding_3_large_1824.pt', 'out/desc/embedding_3_large_55.pt', 'out/train/embedding_3_large_1725993.pt']
model_name = "text-embedding-3-large"
for df, output in zip(data, output_file):
    embed_dataframe_column(df, output, model_name, column="code", batch_size=100)